## **Feature Enineering and Feature Selection**

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("insurance.csv")

In [3]:
df["Age_BMI"] = df["age"] * df["bmi"]

print(df[["age", "bmi", "Age_BMI"]].head())

   age     bmi  Age_BMI
0   19  27.900  530.100
1   18  33.770  607.860
2   28  33.000  924.000
3   33  22.705  749.265
4   32  28.880  924.160


In [4]:
df["BMI_squared"] = df["bmi"] ** 2

print(df[["bmi", "BMI_squared"]].head())

      bmi  BMI_squared
0  27.900   778.410000
1  33.770  1140.412900
2  33.000  1089.000000
3  22.705   515.517025
4  28.880   834.054400


In [5]:
import numpy as np

df["log_charges"] = np.log(df["charges"])

print(df[["charges", "log_charges"]].head())

       charges  log_charges
0  16884.92400     9.734176
1   1725.55230     7.453302
2   4449.46200     8.400538
3  21984.47061     9.998092
4   3866.85520     8.260197


In [6]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import LinearRegression

# Encode categorical variables
df_encoded = pd.get_dummies(
    df,
    columns=["sex", "smoker", "region"],
    drop_first=True
)

X = df_encoded.drop(["charges", "log_charges"], axis=1)
y = df_encoded["charges"]

model = LinearRegression()

rfe = RFE(
    estimator=model,
    n_features_to_select=5
)

rfe.fit(X, y)

selected_features = X.columns[rfe.support_]

print(selected_features)

Index(['bmi', 'children', 'smoker_yes', 'region_southeast',
       'region_southwest'],
      dtype='str')


In [7]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

X_selected = X[selected_features]

X_train, X_test, y_train, y_test = train_test_split(
    X_selected,
    y,
    test_size=0.2, 
    random_state = 42
)

model = LinearRegression()

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred) ** 0.5
r2 = r2_score(y_test, y_pred)

print("MAE :", mae)
print("RMSE:", rmse)
print("R²  :", r2)

MAE : 5453.934865827824
RMSE: 6842.930175073364
R²  : 0.6983827633824445


## **MISSING VALUES PRACTICE**

In [8]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "Age": [20, 25, np.nan, 30, 35]
})

In [9]:
df['Age'] = df['Age'].fillna(df['Age'].mean())

In [10]:
df

,Age
0,20.0
1,25.0
2,27.5
3,30.0
4,35.0


In [11]:
df = pd.DataFrame({
    "Age": [20, 25, np.nan, 30, 35]
})
df['Age'] = df['Age'].fillna(df['Age'].median())

In [12]:
df

,Age
0,20.0
1,25.0
2,27.5
3,30.0
4,35.0


In [13]:
df = pd.DataFrame({
    "Gender": ["Male", "Female", np.nan, "Male", "Female"]
})

df['Gender'] = df['Gender'].fillna(df['Gender'].mode()[0])

In [14]:
df

,Gender
0,Male
1,Female
2,Female
3,Male
4,Female


In [15]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "BMI": [25.2, np.nan, 30.1, 27.5, np.nan]
})

In [16]:
from sklearn.impute import SimpleImputer 
imputer = SimpleImputer(strategy = 'mean')
df['BMI'] = imputer.fit_transform(df[['BMI']])

In [17]:
df

,BMI
0,25.2
1,27.6
2,30.1
3,27.5
4,27.6


In [18]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "BMI": [25, np.nan, 30, 28, np.nan]
})

In [19]:
print("Original Data")
print(df)

print("\nAfter Row Deletion")
print(df.dropna())

print("\nAfter Mean Imputation") 
df['BMI'] = df['BMI'].fillna(df['BMI'].mean()) 
print(df)

Original Data
    BMI
0  25.0
1   NaN
2  30.0
3  28.0
4   NaN

After Row Deletion
    BMI
0  25.0
2  30.0
3  28.0

After Mean Imputation
         BMI
0  25.000000
1  27.666667
2  30.000000
3  28.000000
4  27.666667


## **ENCODING PRACTICE**

In [20]:
import pandas as pd

df = pd.DataFrame({
    "Color": ["Red", "Blue", "Green"],
    "Education": ["High School", "Bachelor's", "Master's"],
    "Purchased": ["Yes", "No", "Yes"]
})

print(df)

   Color    Education Purchased
0    Red  High School       Yes
1   Blue   Bachelor's        No
2  Green     Master's       Yes


In [21]:
from sklearn.preprocessing import LabelEncoder 
encoder = LabelEncoder() 
df['Purchased'] = encoder.fit_transform(df['Purchased'])

In [22]:
from sklearn.preprocessing import OrdinalEncoder
encoder = OrdinalEncoder(categories = [["High School", "Bachelor's", "Master's"]])
df['Education'] = encoder.fit_transform(df[['Education']])

In [23]:
encoded_df = pd.get_dummies(df['Color'])

In [24]:
encoded_df.head()

,Blue,Green,Red
0,False,False,True
1,True,False,False
2,False,True,False


In [25]:
df = pd.concat([df, encoded_df] , axis = 1)

In [26]:
df = df.drop(columns = ['Color'])

In [27]:
df

,Education,Purchased,Blue,Green,Red
0,0.0,1,False,False,True
1,1.0,0,True,False,False
2,2.0,1,False,True,False
